# Generative Information Retrieval for Science

## Creating Database

In [1]:
from pinecone import Pinecone, ServerlessSpec
import os
from dotenv import load_dotenv

load_dotenv()
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

In [25]:
pc.create_index(
    name="science-assistant-index",
    dimension=1536,
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

{
    "name": "science-assistant-index",
    "metric": "cosine",
    "host": "science-assistant-index-iujq88n.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 1536,
    "deletion_protection": "disabled",
    "tags": null
}

## Models to Generate and Verify

In [1]:
from apis.openai_api import queryOpenAI
response = queryOpenAI(prompt = "Why do Europeans like eating cold food even in winter?", sysRole = "You are a helpful direct chatbot which gives informative explainations.")
print(response)

The preference for cold food in Europe, even during winter, can be attributed to several cultural, historical, and practical factors:

1. **Culinary Traditions**: Many European countries have longstanding culinary traditions that include cold dishes. For example, in countries like Italy, Spain, and France, cold cuts, cheeses, and salads are staples that are enjoyed year-round. These dishes are often part of social meals and gatherings.

2. **Preservation Methods**: Historically, before refrigeration became common, people relied on methods like curing, smoking, and pickling to preserve food. These methods often resulted in cold dishes that could be stored for longer periods. This practice has influenced modern eating habits.

3. **Social and Lifestyle Factors**: In many European cultures, meals are social events that can last for several hours. Cold dishes, such as charcuterie boards or antipasti, are often served as part of a leisurely meal, allowing for easy sharing and grazing.

4. *

In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained('cross-encoder/nli-deberta-v3-base')
tokenizer = AutoTokenizer.from_pretrained('cross-encoder/nli-deberta-v3-base')

features = tokenizer(['A man is eating pizza', 'A black race car starts up in front of a crowd of people.'], ['A man eats something', 'A man is driving down a lonely road.'],  padding=True, truncation=True, return_tensors="pt")

model.eval()
with torch.no_grad():
    scores = model(**features).logits
    label_mapping = ['contradiction', 'entailment', 'neutral']
    labels = [label_mapping[score_max] for score_max in scores.argmax(dim=1)]
    print(labels)

['entailment', 'contradiction']


## Populating Database

In [1]:
import arxiv
import os

In [ ]:
search = arxiv.Search(
    query="cat:cs.CL AND \"Large Language Models\"",
    max_results=70,
    sort_by=arxiv.SortCriterion.Relevance
)

os.makedirs("papers", exist_ok=True)
for result in search.results():
    try:
        path = result.download_pdf(dirpath="papers")
        print(f"Downloaded: {result.title}")
    except:
        continue

C:\Users\Devansh\AppData\Local\Temp\ipykernel_18508\4221297801.py:11: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in search.results():


Downloaded: Unmasking the Shadows of AI: Investigating Deceptive Capabilities in Large Language Models
Downloaded: Is Self-knowledge and Action Consistent or Not: Investigating Large Language Model's Personality
Downloaded: Large Language Models Lack Understanding of Character Composition of Words
Downloaded: Making Large Language Models Better Reasoners with Alignment
Downloaded: Self-Cognition in Large Language Models: An Exploratory Study
Downloaded: A Critical Review of Causal Reasoning Benchmarks for Large Language Models
Downloaded: TEAL: Tokenize and Embed ALL for Multi-modal Large Language Models
Downloaded: Not All Experts are Equal: Efficient Expert Pruning and Skipping for Mixture-of-Experts Large Language Models
Downloaded: Classifying German Language Proficiency Levels Using Large Language Models
Downloaded: Large Language Models in Ambulatory Devices for Home Health Diagnostics: A case study of Sickle Cell Anemia Management
Downloaded: Attacks on Third-Party APIs of Large

In [6]:
from pathlib import Path

folder_path = Path("papers")
# Counts only files, excluding subdirectories
file_count = sum(1 for x in folder_path.iterdir() if x.is_file())
print(f"Total files: {file_count}")

Total files: 69


In [34]:
import time
import os
import arxiv

def get_clean_metadata(file_path):
    # Extracts '2305.16300' from 'papers/2305.16300v1.pdf'
    path_split = file_path.split(".")
    arxiv_id = path_split[0] +"."+ path_split[1]
    attempt = 0
    while attempt != 2:
        try:
            search = arxiv.Search(id_list=[arxiv_id])
            time.sleep(5)
            paper = next(search.results())
            
            return {
                "title": paper.title,
                "author": "; ".join([a.name for a in paper.authors]),
                "year": paper.published.year,
                "arxiv_id": arxiv_id
            }
        except Exception:
            if attempt == 2:
                print(f"Couldn't find data for {file_path}")
                return {"title": "Unknown", "author": "Unknown", "year": "Unknown", "arxiv_id": arxiv_id}
            else:
                attempt += 1


In [35]:
get_clean_metadata("2403.09676v1.Unmasking_the_Shadows_of_AI__Investigating_Deceptive_Capabilities_in_Large_Language_Models.pdf")

C:\Users\Devansh\AppData\Local\Temp\ipykernel_13816\2007109412.py:14: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  paper = next(search.results())


{'title': 'Unmasking the Shadows of AI: Investigating Deceptive Capabilities in Large Language Models',
 'author': 'Linge Guo',
 'year': 2024,
 'arxiv_id': '2403.09676v1'}

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import time

all_chunks = []
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)

for filename in os.listdir("papers"):
    if filename.endswith(".pdf"):
        file_path = f"papers/{filename}"
        clean_meta = get_clean_metadata(filename)
       
        loader = PyPDFLoader(file_path)
        pages = loader.load() ## Each 'page' becomes a document object with metadata['page']

        for page in pages:
            # Overwrite/Add clean metadata to the LangChain document object
            page.metadata.update(clean_meta)

        chunks = text_splitter.split_documents(pages)
        all_chunks.extend(chunks)

C:\Users\Devansh\AppData\Local\Temp\ipykernel_13816\2007109412.py:14: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  paper = next(search.results())
Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 36 0 (offset 0)
Ignoring wrong pointing object 67 0 (offset 0)
Ignoring wrong pointing object 106 0 (offset 0)
Ignoring wrong pointing object 108 0 (offset 0)
Ignori

In [37]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

# Initialize embeddings (requires OPENAI_API_KEY)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Create the store and upload everything at once
vector_store = PineconeVectorStore.from_documents(
    all_chunks, 
    embeddings, 
    index_name="science-assistant-index"
)

Trying to retreive from database

In [38]:
from langchain_pinecone import PineconeVectorStore
from langchain_openai import OpenAIEmbeddings

# Must be the SAME model used during upsert!
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Connect to the existing index
vector_store = PineconeVectorStore(
    index_name="science-assistant-index", 
    embedding=embeddings,
    text_key="text" # This matches the key where your page content is stored
)

In [39]:
query = "What are the primary findings regarding LLM latency in space?"

# Retrieve top 4 most relevant chunks
docs = vector_store.similarity_search(query, k=4)

# Print the results with Provenance (Source and Page)
for i, doc in enumerate(docs):
    print(f"--- Result {i+1} ---")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")
    print(f"Page: {doc.metadata.get('page', 'Unknown')}")
    print(f"Snippet: {doc.page_content[:200]}...")
    print("\n")

--- Result 1 ---
Source: papers/2507.12472v1.A_Survey_of_AIOps_in_the_Era_of_Large_Language_Models.pdf
Page: 23.0
Snippet: with LLMs can be prohibitive, especially when real-time or near-real-time responses are required.
This poses significant hurdles, particularly for small to medium-sized enterprises or in scenarios
whe...


--- Result 2 ---
Source: papers/2501.09223v2.Foundations_of_Large_Language_Models.pdf
Page: 239.0
Snippet: due to differences in task types and prompt lengths. Such variability renders simple static load
balancing approaches ineffective, and so we need to use finer-grained strategies that can adapt to
runt...


--- Result 3 ---
Source: papers/2501.09223v2.Foundations_of_Large_Language_Models.pdf
Page: 239.0
Snippet: details of LLM serving. For related concepts and techniques, readers may refer to relevant open-
source systems (such as vLLM 4, TensorRT-LLM5 and TGI 6) and papers [Pope et al., 2023; Li
et al., 2024...


--- Result 4 ---
Source: papers/2507.12472v1.

In [40]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# When you invoke the retriever, it handles the embedding of the question for you
relevant_docs = retriever.invoke(query)

In [41]:
for i in range(len(relevant_docs)):
    print(relevant_docs[i])
    print()

page_content='with LLMs can be prohibitive, especially when real-time or near-real-time responses are required.
This poses significant hurdles, particularly for small to medium-sized enterprises or in scenarios
where computational resources are constrained.
Among the tasks of AIOps, the computational overhead of LLMs is most critical for failure
perception tasks. These steps theoretically need to run continuously during the operation of a
software system and require high real-time performance. For example, if the anomaly detection time
window is set to 10 seconds, meaning an anomaly detection process is triggered every 10 seconds,
the anomaly detection model must infer results within 1 second to promptly notify OCEs in case
of anomalies. Some non-LLM-based approaches have focused on addressing this issue [ 49, 74],
but currently, no LLM-based work has adequately tackled this problem. However, this issue is
becoming increasingly significant in the LLM era.' metadata={'arxiv_id': '2507.1

In [42]:
print(relevant_docs)

[Document(id='d121d585-a3e0-4106-978d-02b3b6f7f622', metadata={'arxiv_id': '2507.12472v1', 'arxivid': 'https://arxiv.org/abs/2507.12472v1', 'author': 'Lingzhe Zhang; Tong Jia; Mengxi Jia; Yifan Wu; Aiwei Liu; Yong Yang; Zhonghai Wu; Xuming Hu; Philip S. Yu; Ying Li', 'creationdate': '2025-07-18T00:00:21+00:00', 'creator': 'arXiv GenPDF (tex2pdf:)', 'doi': 'https://doi.org/10.48550/arXiv.2507.12472', 'keywords': 'Large Language Model, AIOps, Metrics, Logs, Time Series, Incident Report, Failure Perception, Anomaly Detection, Root Cause Analysis, Assisted Remediation', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'moddate': '2025-07-18T00:00:21+00:00', 'page': 23.0, 'page_label': '24', 'producer': 'pikepdf 8.15.1', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'source': 'papers/2507.12472v1.A_Survey_of_AIOps_in_the_Era_of_Large_Language_Models.pdf', 'subject': '-  Software and its engineering  ->  Maintain

## Creating Questions

In [1]:
questions = [
    "What is the specific architecture used in the \"Attention is All You Need\" paper?",
    "How does the \"text-embedding-3-small\" model differ from previous iterations in terms of dimensions?",
    "Define the concept of \"Temperature\" in the context of LLM decoding.",
    "According to ArXiv papers on RAG, what is the \"Lost in the Middle\" phenomenon?",
    "What are the primary differences between Supervised Fine-Tuning (SFT) and RLHF?",
    "Compare the scaling laws of Chinchilla versus GPT-3 based on the provided context.",
    "Does the context suggest that increasing model parameters always leads to better reasoning?",
    "Contrast \"Chain of Thought\" prompting with \"Tree of Thoughts\" methodology.",
    "Based on the papers, is LoRA (Low-Rank Adaptation) more efficient than full fine-tuning for all model sizes?",
    "Identify two specific risks of \"Hallucination\" mentioned in recent NLP survey papers.",
    "What does the acronym \"LLM\" stand for?",
    "Who are the original authors of the \"Attention is All You Need\" paper?",
    "What is the main purpose of a \"Tokenizer\" in NLP?",
    "Is GPT-4 considered a \"Transformer-based\" model?",
    "List three common applications of Large Language Models mentioned in the text.",
    "List three authors who have published significant work on \"Sparse Attention.\"",
    "What specific page in the retrieved context discusses the \"Softmax\" function?",
    "How is \"Perplexity\" mathematically defined in the provided snippets?",
    "Using only the provided sources, explain the role of \"Positional Encodings.\""
]

## Evaluation

Metrics
1) Relevant Statement: This ratio measures the fraction of relevant statements in the answer text in relation to the total number of statements.
2) Uncited Sources: This ratio metric measures the fraction of sources that are cited in the answer text in relation to the total number of listed sources.
3) Unsupported Statements: This ratio metric measures the fraction of relevant statements that are not factually supported by any of the listed sources. Any row of the factual support matrix with no checked cell corresponds to an unsupported statement.
4) Citation Accuracy: This ratio metric measures the fraction of statement citations that accurately reflect that a source’s content supports the statement. This metric can be computed by measuring the overlap between the citation and the factual support matrices, and dividing by the number of citations
5) Citation Thoroughness: This ratio metric measures the fraction of accurate citations included in the answer text compared to all possible accurate citations



In [1]:
from generateScientificResponse import generateResponse

c:\Users\Devansh\Downloads\SC4052\Course Project\Project Code\.cloudComputingProject\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
context, response = generateResponse(query= "How do LLMs work?")
for source in context:
    print(source)
print()
print(response)

{'source': '2507.08425v1', 'author': 'Lu Xiang et al.', 'title': 'A Survey of Large Language Models in Discipline-specific Research: Challenges, Methods and Opportunities', 'page': 0.0, 'content': 'three primary objectives: (1) to explore the key\ntechnical methods used to adapt LLMs to meet\nthe specific needs of various disciplines; (2) to ex-\namine practical applications of LLMs in diverse\ndisciplines, including mathematics, physics, chem-\nistry, biology, and the humanities and social sci-\nences, with an emphasis on how LLMs address\nunique challenges; (3) to outline the challenge and\nopportunities for interdisciplinary collaboration fa-\ncilitated by LLMs.\nThe remainder of this article is organised as fol-\nlows. Section 2 provides the preliminaries of LLMs\nand the challenges related to applying LLMs into\nvarious disciplines. Following that, the taxonomy\nof LLM techniques for different disciplines is de-\narXiv:2507.08425v1  [cs.CL]  11 Jul 2025. duce human-like language w

In [4]:
from generateScientificResponse import extractStatements

consolidatedStatements = extractStatements(response=response)
print(consolidatedStatements)

['Large Language Models (LLMs) operate by leveraging advanced machine learning techniques to generate human-like text based on the input they receive. They are trained on vast datasets, which allows them to understand and produce language with remarkable fluency. This training enables LLMs to perform various tasks, including general knowledge question answering and complex problem-solving, thanks to their strong reasoning capabilities [Source: 2507.08425v1, Page: 0.0].,', 'From a technical standpoint, LLMs utilize several key methodologies to enhance their adaptability and effectiveness in specific contexts. These include supervised fine-tuning, which adjusts the model based on labeled data; retrieval-augmented generation, which combines the models generative capabilities with external information retrieval; agent-based approaches, which allow LLMs to act autonomously in certain tasks; and tool-use integration, which enables them to interact with other software or databases [Source: 25

In [1]:
from generateScientificResponse import getScientificResponse

response, changes, stats = getScientificResponse(query= "How do LLMs work?")

c:\Users\Devansh\Downloads\SC4052\Course Project\Project Code\.cloudComputingProject\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Generated Base Response
Extracted Statements
['Large Language Models (LLMs) operate by leveraging advanced machine learning techniques to generate human-like text based on the input they receive. They are designed to produce fluent language and can effectively answer general knowledge questions due to their ability to process and utilize vast amounts of information. This capability is underpinned by their strong reasoning skills, which enable them to tackle complex problems and provide informed insights across various domains, including mathematics, physics, biology, and the humanities [Source: 2507.08425v1, Page: 0.0].,', 'From a technical perspective, LLMs employ several key methodologies to enhance their adaptability and effectiveness in specific contexts. These include supervised fine-tuning, which involves training the model on labeled data to improve its performance on particular tasks; retrieval-augmented generation, which allows the model to access external information to enric

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 15159.23it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Lables: ['neutral', 'neutral', 'neutral']
Evaluated Response


In [2]:
print(response)

Large Language Models (LLMs) operate by leveraging advanced machine learning techniques to generate human-like text based on the input they receive. They are designed to produce fluent language and can effectively answer general knowledge questions due to their ability to process and utilize vast amounts of information. This capability is underpinned by their strong reasoning skills, which enable them to tackle complex problems and provide informed insights across various domains, including mathematics, physics, biology, and the humanities [Source: 2507.08425v1, Page: 0.0].,
From a technical perspective, LLMs employ several key methodologies to enhance their adaptability and effectiveness in specific contexts. These include supervised fine-tuning, which involves training the model on labeled data to improve its performance on particular tasks; retrieval-augmented generation, which allows the model to access external information to enrich its responses; agent-based approaches, which ena

In [3]:
print(changes)

Statements changed:



In [4]:
print(stats)

{'relevant': 1.0, 'uncited': 1.0, 'unsupported': 0.0, 'citationAcc': 0.3333333333333333, 'citationTh': 1.0}


In [2]:
from generateScientificResponse import getScientificResponse
successfulPrompts = 0
for i in range(len(questions)):
    question = questions[i]
    try:
        findings = {
            "relevant": 0.0,
            "uncited": 0.0,
            "unsupported": 0.0,
            "citationAcc": 0.0,
            "citationTh": 0.0
        }
        response, changes, stats = getScientificResponse(query=question)
        print(f"Finished question: {question}")
        if stats != {}:
            successfulPrompts += 1
            findings["relevant"] += stats['relevant']
            findings["uncited"] += stats['uncited']
            findings["unsupported"] += stats['unsupported']
            findings["citationAcc"] += stats['citationAcc']
            findings["citationTh"] += stats['citationTh']
    except:
        continue

c:\Users\Devansh\Downloads\SC4052\Course Project\Project Code\.cloudComputingProject\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 12911.25it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11682.81it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
de

Finished question: Does the context suggest that increasing model parameters always leads to better reasoning?


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 12186.96it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11259.88it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Finished question: Based on the papers, is LoRA (Low-Rank Adaptation) more efficient than full fine-tuning for all model sizes?


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 13049.26it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Finished question: Is GPT-4 considered a "Transformer-based" model?


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 11574.60it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Finished question: List three common applications of Large Language Models mentioned in the text.


Loading weights: 100%|██████████| 202/202 [00:00<00:00, 14031.95it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 12788.67it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 14141.56it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/n

Finished question: Using only the provided sources, explain the role of "Positional Encodings."


In [3]:
for key in findings.keys():
    findings[key] /= successfulPrompts
    print(f"{key}: {findings[key]}")

relevant: 0.16666666666666666
uncited: 0.16666666666666666
unsupported: 0.0
citationAcc: 0.03333333333333333
citationTh: 0.08333333333333333


In [4]:
print(successfulPrompts)

6
